In [3]:
# YOUR LOCAL PATHS
BASE_PATH = r"C:\Users\dhank\Downloads\deepfake-detection-challenge"
TRAIN_PATH = os.path.join(BASE_PATH, "train_sample_videos")
TEST_PATH = os.path.join(BASE_PATH, "test_videos")
METADATA_PATH = os.path.join(BASE_PATH, "train_sample_videos", "metadata.json")

print(f"Train videos: {len(os.listdir(TRAIN_PATH))}")
print(f"Test videos: {len(os.listdir(TEST_PATH))}")
# Output: Train videos: 400, Test videos: 400 ✓


Train videos: 401
Test videos: 400


In [2]:
import os

In [4]:
import pandas as pd
import json

# Load YOUR metadata.json (has REAL/FAKE labels)
train_sample_metadata = pd.read_json(METADATA_PATH).T
train_sample_metadata['label'] = train_sample_metadata['label'].map({'REAL': 0, 'FAKE': 1})
print(train_sample_metadata.head())
print(f"FAKE: {sum(train_sample_metadata['label'])}, REAL: {len(train_sample_metadata) - sum(train_sample_metadata['label'])}")


                label  split        original
aagfhgtpmv.mp4      1  train  vudstovrck.mp4
aapnvogymq.mp4      1  train  jdubbvfswz.mp4
abarnvbtwb.mp4      0  train            None
abofeumbvv.mp4      1  train  atvmxvwyns.mp4
abqwwspghj.mp4      1  train  qzimuostzz.mp4
FAKE: 323, REAL: 77


In [32]:
!pip install --upgrade pip

!pip install tensorflow==2.15.0
!pip install opencv-python
!pip install numpy pandas scikit-learn

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 16.4 MB/s  0:00:00


ERROR: To modify pip, please run the following command:
C:\Users\dhank\OneDrive\Documents\coding\venv\python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.15.0


In [40]:
!pip uninstall tensorflow -y
!pip install tensorflow-cpu==2.15.0

Found existing installation: tensorflow 2.21.0
Uninstalling tensorflow-2.21.0:
  Successfully uninstalled tensorflow-2.21.0


You can safely remove it manually.
You can safely remove it manually.
ERROR: Could not find a version that satisfies the requirement tensorflow-cpu==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow-cpu==2.15.0


In [41]:
import tensorflow as tf
print(tf.__version__)

AttributeError: module 'tensorflow' has no attribute '__version__'

In [39]:
!pip install tensorflow 
import tensorflow as tf
print(tf.__version__)
from tensorflow import keras

ImportError: DLL load failed while importing _pywrap_tensorflow_lite_metrics_wrapper: The specified procedure could not be found.

In [3]:
# train_on_your_data.py - RUN THIS FIRST
import os, cv2, numpy as np, pandas as pd
from tensorflow import keras  # Direct import from tensorflow

from sklearn.model_selection import train_test_split

# YOUR PATHS
BASE_PATH = r"C:\Users\dhank\Downloads\deepfake-detection-challenge"
TRAIN_PATH = os.path.join(BASE_PATH, "train_sample_videos")
METADATA_PATH = os.path.join(TRAIN_PATH, "metadata.json")

IMG_SIZE = 224
MAX_SEQ_LENGTH = 20
NUM_FEATURES = 2048

# 1. Load YOUR metadata
train_sample_metadata = pd.read_json(METADATA_PATH).T
train_sample_metadata['label'] = train_sample_metadata['label'].map({'REAL': 0, 'FAKE': 1})

# 2. Core functions (unchanged)
def square_crop_frame(image):
    h, w = image.shape[:2]
    size = min(h, w)
    start_x, start_y = (w - size) // 2, (h - size) // 2
    return image[start_y:start_y+size, start_x:start_x+size]

def process_video_frames(video_path, max_frames=MAX_SEQ_LENGTH):
    cap = cv2.VideoCapture(video_path)
    frames = []
    for _ in range(max_frames):
        ret, frame = cap.read()
        if not ret: break
        frame = square_crop_frame(frame)
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    return np.array(frames)

def build_feature_extractor():
    base_model = keras.applications.ResNet50(
        weights="imagenet", include_top=False, pooling="avg",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = keras.applications.resnet50.preprocess_input(inputs)
    outputs = base_model(x)
    return keras.Model(inputs, outputs)

def extract_features(dataframe, directory):
    """Extract features FROM YOUR LOCAL VIDEOS"""
    feature_extractor = build_feature_extractor()
    total_videos = len(dataframe)
    video_features = np.zeros((total_videos, MAX_SEQ_LENGTH, NUM_FEATURES))
    video_masks = np.zeros((total_videos, MAX_SEQ_LENGTH), dtype=bool)
    labels = dataframe['label'].values
    
    for idx, video_file in enumerate(dataframe.index):
        video_path = os.path.join(directory, video_file)
        frames = process_video_frames(video_path)
        
        # Pad/truncate
        if len(frames) < MAX_SEQ_LENGTH:
            pad_frames = np.repeat(frames[-1:], MAX_SEQ_LENGTH - len(frames), axis=0)
            frames = np.concatenate([frames, pad_frames])
        else:
            frames = frames[:MAX_SEQ_LENGTH]
        
        # Extract features
        for i in range(MAX_SEQ_LENGTH):
            feat = feature_extractor.predict(
                np.expand_dims(frames[i], 0), verbose=0
            )[0]
            video_features[idx, i] = feat
        video_masks[idx, :len(frames)] = True
        print(f"Processed {idx+1}/{total_videos}: {video_file}")
    
    return (video_features, video_masks), labels

# 3. SPLIT YOUR DATA
Train_set, Test_set = train_test_split(
    train_sample_metadata, test_size=0.1, 
    random_state=42, stratify=train_sample_metadata['label']
)

# 4. EXTRACT FEATURES FROM YOUR LOCAL VIDEOS
print("🔄 Extracting features from YOUR train videos...")
train_data, train_labels = extract_features(Train_set, TRAIN_PATH)
print("🔄 Extracting features from YOUR test videos...")
test_data, test_labels = extract_features(Test_set, TRAIN_PATH)



🔄 Extracting features from YOUR train videos...
Processed 1/360: curpwogllm.mp4
Processed 2/360: btmsngnqhv.mp4
Processed 3/360: cqhngvpgyi.mp4
Processed 4/360: boovltmuwi.mp4
Processed 5/360: dzqwgqewhu.mp4
Processed 6/360: dsdoseflas.mp4
Processed 7/360: cwrtyzndpx.mp4
Processed 8/360: dqnyszdong.mp4
Processed 9/360: abqwwspghj.mp4
Processed 10/360: bdbhekrrwo.mp4
Processed 11/360: dzieklokdr.mp4
Processed 12/360: avssvvsdhz.mp4
Processed 13/360: cknyxaqouy.mp4
Processed 14/360: adhsbajydo.mp4
Processed 15/360: cffffbcywc.mp4
Processed 16/360: drgjzlxzxj.mp4
Processed 17/360: cycacemkmt.mp4
Processed 18/360: akxoopqjqz.mp4
Processed 19/360: agrmhtjdlk.mp4
Processed 20/360: ebchwmwayp.mp4
Processed 21/360: akvmwkdyuv.mp4
Processed 22/360: eprybmbpba.mp4
Processed 23/360: dtbpmdqvao.mp4
Processed 24/360: alaijyygdv.mp4
Processed 25/360: ckkuyewywx.mp4
Processed 26/360: aelfnikyqj.mp4
Processed 27/360: bddjdhzfze.mp4
Processed 28/360: ajwpjhrbcv.mp4
Processed 29/360: ahbweevwpv.mp4
Proc

In [5]:
# 2a. Build a global feature extractor
feature_extractor = build_feature_extractor()  # <-- ADD THIS BEFORE calling extract_features

# 4. EXTRACT FEATURES FROM YOUR LOCAL VIDEOS
print("🔄 Extracting features from YOUR train videos...")
train_data, train_labels = extract_features(Train_set, TRAIN_PATH)
print("🔄 Extracting features from YOUR test videos...")
test_data, test_labels = extract_features(Test_set, TRAIN_PATH)

# 5. BUILD & TRAIN MODEL
model = build_gru_model()

history = model.fit(
    [train_data[0], train_data[1]],
    train_labels,
    validation_data=([test_data[0], test_data[1]], test_labels),
    epochs=10,
    batch_size=8,
    verbose=1
)

# 6. SAVE WEIGHTS
os.makedirs("production_weights", exist_ok=True)

feature_extractor.save_weights("production_weights/feature_extractor_weights.h5")
model.save_weights("production_weights/gru_model_weights.h5")

print("✅ SAVED PRODUCTION WEIGHTS!")

🔄 Extracting features from YOUR train videos...
Processed 1/360: curpwogllm.mp4
Processed 2/360: btmsngnqhv.mp4
Processed 3/360: cqhngvpgyi.mp4
Processed 4/360: boovltmuwi.mp4
Processed 5/360: dzqwgqewhu.mp4
Processed 6/360: dsdoseflas.mp4
Processed 7/360: cwrtyzndpx.mp4
Processed 8/360: dqnyszdong.mp4
Processed 9/360: abqwwspghj.mp4
Processed 10/360: bdbhekrrwo.mp4
Processed 11/360: dzieklokdr.mp4
Processed 12/360: avssvvsdhz.mp4
Processed 13/360: cknyxaqouy.mp4
Processed 14/360: adhsbajydo.mp4
Processed 15/360: cffffbcywc.mp4
Processed 16/360: drgjzlxzxj.mp4
Processed 17/360: cycacemkmt.mp4
Processed 18/360: akxoopqjqz.mp4
Processed 19/360: agrmhtjdlk.mp4
Processed 20/360: ebchwmwayp.mp4
Processed 21/360: akvmwkdyuv.mp4
Processed 22/360: eprybmbpba.mp4
Processed 23/360: dtbpmdqvao.mp4
Processed 24/360: alaijyygdv.mp4
Processed 25/360: ckkuyewywx.mp4


KeyboardInterrupt: 

In [8]:
# Make sure feature_extractor exists globally
feature_extractor = build_feature_extractor()  # <--- ADD THIS

# Build & train GRU model
model = build_gru_model()

history = model.fit(
    [train_data[0], train_data[1]],
    train_labels,
    validation_data=([test_data[0], test_data[1]], test_labels),
    epochs=10,
    batch_size=8,
    verbose=1
)

# Save weights
os.makedirs("production_weights", exist_ok=True)

feature_extractor.save_weights("production_weights/feature_extractor_weights.h5")
model.save_weights("production_weights/gru_model_weights.h5")

print("✅ SAVED PRODUCTION WEIGHTS!")

Epoch 1/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 23s 68ms/step - accuracy: 0.8083 - loss: 1.9346 - val_accuracy: 0.8000 - val_loss: 1.3357
Epoch 2/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.8056 - loss: 1.0771 - val_accuracy: 0.8000 - val_loss: 0.9476
Epoch 3/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.8056 - loss: 0.8398 - val_accuracy: 0.8000 - val_loss: 0.8315
Epoch 4/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.8083 - loss: 0.7588 - val_accuracy: 0.8000 - val_loss: 0.7188
Epoch 5/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.8111 - loss: 0.6866 - val_accuracy: 0.8000 - val_loss: 0.6747
Epoch 6/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.8056 - loss: 0.6496 - val_accuracy: 0.8000 - val_loss: 0.6395
Epoch 7/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.8083 - loss: 0.6036 - val_accuracy: 0.8000 - val_loss: 0.6048
Epoch 8/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.8083 - loss: 0.5828 - val_accuracy: 0.8000 - 

ValueError: The filename must end in `.weights.h5`. Received: filepath=production_weights/feature_extractor_weights.h5

In [9]:
# Save weights
os.makedirs("production_weights", exist_ok=True)

feature_extractor.save_weights("production_weights/feature_extractor_weights.weights.h5")
model.save_weights("production_weights/gru_model_weights.weights.h5")

print("✅ SAVED PRODUCTION WEIGHTS!")

✅ SAVED PRODUCTION WEIGHTS!


In [10]:
# Make sure feature_extractor exists globally
feature_extractor = build_feature_extractor()  # <--- ADD THIS

# Build & train GRU model
model = build_gru_model()

history = model.fit(
    [train_data[0], train_data[1]],
    train_labels,
    validation_data=([test_data[0], test_data[1]], test_labels),
    epochs=10,
    batch_size=8,
    verbose=1
)

# Save weights
os.makedirs("production_weights", exist_ok=True)

feature_extractor.save_weights("production_weights/feature_extractor_weights.h5")
model.save_weights("production_weights/gru_model_weights.h5")

print("✅ SAVED PRODUCTION WEIGHTS!")

Epoch 1/10


KeyboardInterrupt: 

In [7]:
def build_gru_model():
    frame_input = keras.layers.Input((MAX_SEQ_LENGTH, NUM_FEATURES))
    mask_input = keras.layers.Input((MAX_SEQ_LENGTH,), dtype="bool")

    x = keras.layers.Bidirectional(
        keras.layers.GRU(
            16,
            return_sequences=True,
            kernel_regularizer=keras.regularizers.l2(0.01)
        )
    )(frame_input, mask=mask_input)

    x = keras.layers.Bidirectional(
        keras.layers.GRU(
            8,
            kernel_regularizer=keras.regularizers.l2(0.01)
        )
    )(x)

    x = keras.layers.Dropout(0.5)(x)
    x = keras.layers.Dense(8, activation="relu")(x)
    output = keras.layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model([frame_input, mask_input], output)

    # ✅ Correct compile
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model




In [25]:
# =============================================================================
# COMPLETE IMPORTS FOR DEEPFAKE DETECTION PIPELINE
# =============================================================================

# Core Python Libraries
import os                           # File/Directory operations
import sys                          # System parameters
import json                         # JSON metadata parsing
import tempfile                     # Temporary file handling for Streamlit

# Data Processing & Scientific Computing
import numpy as np                  # Array operations, math
import pandas as pd                 # DataFrame for metadata handling

# Computer Vision & Video Processing
import cv2                          # Video frame extraction, image processing

# Machine Learning & Deep Learning
import tensorflow as tf             # TensorFlow core
from tensorflow import keras        # High-level Keras API
from tensorflow.keras import layers # Neural network layers
from tensorflow.keras import models # Model building
from tensorflow.keras import callbacks  # Training callbacks

# Data Splitting
from sklearn.model_selection import train_test_split  # Train/test split

# Visualization (Optional for training/debugging)
import matplotlib.pyplot as plt     # Plotting training history

# Streamlit Web App (Production)
import streamlit as st              # Web interface

# Image Handling (Streamlit video display)
from PIL import Image               # Image processing for Streamlit

# Warnings suppression (clean output)
import warnings
warnings.filterwarnings('ignore')

# Display utilities (Optional for Jupyter)
from IPython.display import HTML, display  # Jupyter video embedding

print("✅ All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"OpenCV version: {cv2.__version__}")


ImportError: DLL load failed while importing _pywrap_tensorflow_lite_metrics_wrapper: The specified procedure could not be found.

In [23]:
!pip install tensorflow==2.15.0
!pip install opencv-python
!pip install numpy pandas scikit-learn

ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.15.0


In [ ]:
# app.py - Uses YOUR trained weights
import streamlit as st
import tensorflow as tf
import numpy as np, cv2, os
from PIL import Image
import tempfile

st.set_page_config(page_title="Deepfake Detector", layout="wide")
st.title("🕵️ Deepfake Detector - Trained on YOUR Dataset")

IMG_SIZE, MAX_SEQ_LENGTH, NUM_FEATURES = 224, 20, 2048

# YOUR FUNCTIONS (same as training)
def square_crop_frame(image):
    h, w = image.shape[:2]
    size = min(h, w)
    start_x, start_y = (w - size) // 2, (h - size) // 2
    return image[start_y:start_y+size, start_x:start_x+size]

def process_video_frames(video_path, max_frames=MAX_SEQ_LENGTH):
    cap = cv2.VideoCapture(video_path)
    frames = []
    for _ in range(max_frames):
        ret, frame = cap.read()
        if not ret: break
        frame = square_crop_frame(frame)
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    return np.array(frames)

@st.cache_resource
def load_your_models():
    """Load YOUR trained weights"""
    # Build architectures
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet", include_top=False, pooling="avg",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.resnet50.preprocess_input(inputs)
    feature_extractor = tf.keras.Model(inputs, base_model(x))
    
    frame_input = tf.keras.layers.Input((MAX_SEQ_LENGTH, NUM_FEATURES))
    mask_input = tf.keras.layers.Input((MAX_SEQ_LENGTH,), dtype="bool")
    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(16, return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(0.01)))(frame_input, mask=mask_input)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(8, kernel_regularizer=tf.keras.regularizers.l2(0.01)))(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(8, "relu")(x)
    output = tf.keras.layers.Dense(1, "sigmoid")(x)
    model = tf.keras.Model([frame_input, mask_input], output)
    
    # Load YOUR weights
    feature_extractor.load_weights("production_weights/feature_extractor_weights.h5")
    model.load_weights("production_weights/gru_model_weights.h5")
    model.compile("binary_crossentropy", "adam", ["accuracy"])
    
    return feature_extractor, model

# Load YOUR trained models
feature_extractor, model = load_your_models()
st.success("✅ Loaded YOUR trained models!")

# UPLOAD ANY MP4
uploaded_file = st.file_uploader("📤 Upload MP4", type=['mp4'])

if uploaded_file:
    tfile = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4')
    tfile.write(uploaded_file.read())
    video_path = tfile.name
    
    col1, col2 = st.columns([3,1])
    with col1: st.video(video_path)
    with col2: st.info(f"**{uploaded_file.name}**\n{uploaded_file.size/1024/1024:.1f}MB")
    
    if st.button("🔍 DETECT", type="primary"):
        with st.spinner("Processing YOUR video..."):
            frames = process_video_frames(video_path)
            
            # Pad to 20 frames
            if len(frames) < MAX_SEQ_LENGTH:
                pad = np.repeat(frames[-1:], MAX_SEQ_LENGTH-len(frames), 0)
                frames = np.concatenate([frames, pad])
            else:
                frames = frames[:MAX_SEQ_LENGTH]
            
            # Feature extraction
            frame_features = np.zeros((1, MAX_SEQ_LENGTH, NUM_FEATURES))
            for i in range(MAX_SEQ_LENGTH):
                feat = feature_extractor.predict(np.expand_dims(frames[i], 0), verbose=0)
                frame_features[0,i] = feat[0]
            
            pred = model.predict([frame_features, np.ones((1,MAX_SEQ_LENGTH),bool)], verbose=0)[0][0]
        
        # Results
        col1,col2,col3 = st.columns(3)
        with col1: st.metric("Fake %", f"{pred:.1%}")
        with col2: 
            st.markdown("🚨 **FAKE**" if pred>=0.5 else "✅ **REAL**", 
                       unsafe_allow_html=True)
        with col3: st.metric("Confidence", f"{max(pred,1-pred):.1%}")
    
    os.unlink(video_path)

st.sidebar.markdown("**Trained on YOUR DFDC dataset**")


2026-03-21 20:00:04.626 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:00:04.632 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:00:05.230 
  command:

    streamlit run c:\Users\dhank\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-21 20:00:05.232 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:00:05.236 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:00:05.255 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:00:05.261 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'production_weights/feature_extractor_weights.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [12]:
# app.py - Deepfake Detector
import streamlit as st
import tensorflow as tf
import numpy as np, cv2, os
from PIL import Image
import tempfile

# --- Page config ---
st.set_page_config(page_title="Deepfake Detector", layout="wide")
st.title("🕵️ Deepfake Detector - Trained on YOUR Dataset")

# --- Constants ---
IMG_SIZE = 224
MAX_SEQ_LENGTH = 20
NUM_FEATURES = 2048

# --- Functions ---
def square_crop_frame(image):
    h, w = image.shape[:2]
    size = min(h, w)
    start_x, start_y = (w - size) // 2, (h - size) // 2
    return image[start_y:start_y+size, start_x:start_x+size]

def process_video_frames(video_path, max_frames=MAX_SEQ_LENGTH):
    cap = cv2.VideoCapture(video_path)
    frames = []
    for _ in range(max_frames):
        ret, frame = cap.read()
        if not ret: break
        frame = square_crop_frame(frame)
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    return np.array(frames)

@st.cache_resource
def load_your_models():
    """Load YOUR trained weights with robust paths"""
    # Base ResNet50 feature extractor
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet", include_top=False, pooling="avg",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.resnet50.preprocess_input(inputs)
    feature_extractor = tf.keras.Model(inputs, base_model(x))
    
    # GRU classifier
    frame_input = tf.keras.layers.Input((MAX_SEQ_LENGTH, NUM_FEATURES))
    mask_input = tf.keras.layers.Input((MAX_SEQ_LENGTH,), dtype="bool")
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(16, return_sequences=True,
                            kernel_regularizer=tf.keras.regularizers.l2(0.01))
    )(frame_input, mask=mask_input)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(8, kernel_regularizer=tf.keras.regularizers.l2(0.01))
    )(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(8, activation="relu")(x)
    output = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    model = tf.keras.Model([frame_input, mask_input], output)
    
    # --- Load weights using absolute paths ---
    base_dir = os.path.dirname(__file__)  # Folder where app.py is
    weights_dir = os.path.join(base_dir, "production_weights")
    
    feature_path = os.path.join(weights_dir, "feature_extractor_weights.h5")
    model_path = os.path.join(weights_dir, "gru_model_weights.h5")
    
    if not os.path.exists(feature_path):
        st.error(f"❌ Missing {feature_path}")
    else:
        feature_extractor.load_weights(feature_path)
    
    if not os.path.exists(model_path):
        st.error(f"❌ Missing {model_path}")
    else:
        model.load_weights(model_path)
    
    model.compile("binary_crossentropy", "adam", ["accuracy"])
    return feature_extractor, model

# --- Load models ---
feature_extractor, model = load_your_models()
st.success("✅ Loaded YOUR trained models!")

# --- Video upload ---
uploaded_file = st.file_uploader("📤 Upload MP4", type=['mp4'])

if uploaded_file:
    tfile = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4')
    tfile.write(uploaded_file.read())
    video_path = tfile.name
    
    col1, col2 = st.columns([3,1])
    with col1: st.video(video_path)
    with col2: st.info(f"**{uploaded_file.name}**\n{uploaded_file.size/1024/1024:.1f}MB")
    
    if st.button("🔍 DETECT", type="primary"):
        with st.spinner("Processing video..."):
            frames = process_video_frames(video_path)
            
            # Pad or truncate to MAX_SEQ_LENGTH
            if len(frames) < MAX_SEQ_LENGTH:
                pad = np.repeat(frames[-1:], MAX_SEQ_LENGTH-len(frames), axis=0)
                frames = np.concatenate([frames, pad])
            else:
                frames = frames[:MAX_SEQ_LENGTH]
            
            # Extract features
            frame_features = np.zeros((1, MAX_SEQ_LENGTH, NUM_FEATURES))
            for i in range(MAX_SEQ_LENGTH):
                feat = feature_extractor.predict(np.expand_dims(frames[i], 0), verbose=0)
                frame_features[0,i] = feat[0]
            
            # Predict
            pred = model.predict([frame_features, np.ones((1,MAX_SEQ_LENGTH), bool)], verbose=0)[0][0]
        
        # --- Show results ---
        col1, col2, col3 = st.columns(3)
        with col1: st.metric("Fake %", f"{pred:.1%}")
        with col2:
            st.markdown("🚨 **FAKE**" if pred >= 0.5 else "✅ **REAL**",
                        unsafe_allow_html=True)
        with col3: st.metric("Confidence", f"{max(pred,1-pred):.1%}")
    
    # Delete temp video
    os.unlink(video_path)

st.sidebar.markdown("**Trained on YOUR DFDC dataset**")

2026-03-21 20:03:55.363 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:03:55.414 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:03:55.415 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:03:55.442 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:03:55.541 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:03:55.543 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:03:55.548 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-21 20:03:55.555 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

NameError: name '__file__' is not defined